# 🔍 Deblur/Unblur Image - Training on Google Colab
**Đồ án TGMT - 3 cấp độ: Face, Scene, ID Card**

Notebook này sẽ:
1. Mount Google Drive & setup project
2. Tải dataset (GoPro cho Scene)
3. Train model SwinIR
4. Lưu model về Google Drive

## 1️⃣ Kiểm tra GPU & Mount Google Drive

In [ ]:
!nvidia-smi
print('---')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2️⃣ Upload project lên Google Drive

**Cách làm:** Zip thư mục `CuoiKy` trên máy → upload lên Google Drive vào thư mục `MyDrive/`

Hoặc dùng Git:

In [ ]:
import os

# === CHỌN 1 TRONG 2 CÁCH ===

# CÁCH 1: Đã upload file zip lên Google Drive
PROJECT_ZIP = '/content/drive/MyDrive/CuoiKy.zip'  # <-- sửa đường dẫn nếu cần
if os.path.exists(PROJECT_ZIP):
    !unzip -q {PROJECT_ZIP} -d /content/
    print('[✓] Đã giải nén project từ Drive')

# CÁCH 2: Clone từ GitHub (nếu đã push code)
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/CuoiKy

# Chuyển vào thư mục project
%cd /content/CuoiKy
!ls

In [ ]:
# Cài dependencies
!pip install -q basicsr lmdb pyyaml tb-nightly tqdm gdown

## 3️⃣ Tải Dataset
### Scene: GoPro Large (~5.6GB)

In [ ]:
import os, shutil, glob

# Tải GoPro dataset từ HuggingFace
os.makedirs('downloads', exist_ok=True)
if not os.path.exists('downloads/GOPRO_Large.zip'):
    !wget -q --show-progress -O downloads/GOPRO_Large.zip \
        'https://huggingface.co/datasets/snah/GOPRO_Large/resolve/main/GOPRO_Large.zip'
    print('[✓] Tải xong GoPro!')
else:
    print('[✓] Đã có GoPro!')

# Giải nén
if not os.path.exists('downloads/GOPRO_Large'):
    !cd downloads && unzip -q GOPRO_Large.zip
    print('[✓] Giải nén xong!')

In [ ]:
# Sắp xếp GoPro vào datasets/scene/
import shutil, glob, os

def collect_images(folder):
    paths = []
    for ext in ('*.png', '*.jpg'):
        paths.extend(glob.glob(os.path.join(folder, '**', ext), recursive=True))
    return sorted(set(paths))

gopro = 'downloads/GOPRO_Large'
scene = 'datasets/scene'

for split in ['train', 'val', 'test']:
    os.makedirs(f'{scene}/{split}/blur', exist_ok=True)
    os.makedirs(f'{scene}/{split}/sharp', exist_ok=True)

# Train
count = 0
for d in sorted(os.listdir(f'{gopro}/train')):
    bp = f'{gopro}/train/{d}/blur'
    sp = f'{gopro}/train/{d}/sharp'
    if not os.path.isdir(bp): continue
    for b, s in zip(sorted(glob.glob(f'{bp}/*.png')), sorted(glob.glob(f'{sp}/*.png'))):
        shutil.copy2(b, f'{scene}/train/blur/{d}_{os.path.basename(b)}')
        shutil.copy2(s, f'{scene}/train/sharp/{d}_{os.path.basename(s)}')
        count += 1
print(f'[✓] Train: {count} pairs')

# Test → chia val/test
scenes = sorted([d for d in os.listdir(f'{gopro}/test') if os.path.isdir(f'{gopro}/test/{d}')])
mid = len(scenes) // 2
for split, scene_list in [('val', scenes[:mid]), ('test', scenes[mid:])]:
    c = 0
    for d in scene_list:
        for b, s in zip(sorted(glob.glob(f'{gopro}/test/{d}/blur/*.png')),
                        sorted(glob.glob(f'{gopro}/test/{d}/sharp/*.png'))):
            shutil.copy2(b, f'{scene}/{split}/blur/{d}_{os.path.basename(b)}')
            shutil.copy2(s, f'{scene}/{split}/sharp/{d}_{os.path.basename(s)}')
            c += 1
    print(f'[✓] {split}: {c} pairs')

print('\n[✓] Scene dataset sẵn sàng!')

### Face: CelebA-HQ + Tạo blur tự động
Tải CelebA-HQ 256x256 từ Kaggle (cần Kaggle API key)

In [ ]:
# === CÁCH 1: Tải từ Kaggle ===
# Upload kaggle.json trước (từ kaggle.com/settings → Create New Token)
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d badasstechie/celebahq-resized-256x256 -p downloads/
# !unzip -q downloads/celebahq-resized-256x256.zip -d downloads/celebahq

# === CÁCH 2: Đã có sẵn trên Google Drive ===
# Nếu đã upload ảnh face sharp lên Drive:
# !cp -r '/content/drive/MyDrive/face_sharp_images/'* datasets/face/train/sharp/

# Kiểm tra
import os
os.makedirs('datasets/face/train/sharp', exist_ok=True)
os.makedirs('datasets/face/train/blur', exist_ok=True)
os.makedirs('datasets/face/val/sharp', exist_ok=True)
os.makedirs('datasets/face/val/blur', exist_ok=True)

count = len(glob.glob('datasets/face/train/sharp/*'))
print(f'Face sharp images: {count}')
if count == 0:
    print('⚠️ Chưa có ảnh! Hãy upload ảnh face vào datasets/face/train/sharp/')

In [ ]:
# Tạo blur cho face dataset (chạy SAU KHI có ảnh sharp)
import glob
if len(glob.glob('datasets/face/train/sharp/*')) > 0:
    !python scripts/data_preparation/create_blur_dataset.py \
        --input datasets/face/train/sharp \
        --output datasets/face/train/blur \
        --task face
    # Tạo val set (lấy 10% từ train)
    import random
    all_sharp = sorted(glob.glob('datasets/face/train/sharp/*'))
    val_count = max(50, len(all_sharp) // 10)
    random.seed(42)
    val_imgs = random.sample(all_sharp, val_count)
    for img in val_imgs:
        name = os.path.basename(img)
        shutil.copy2(img, f'datasets/face/val/sharp/{name}')
        blur_img = f'datasets/face/train/blur/{name}'
        if os.path.exists(blur_img):
            shutil.copy2(blur_img, f'datasets/face/val/blur/{name}')
    print(f'[✓] Face: train={len(all_sharp)}, val={val_count}')
else:
    print('⚠️ Chưa có ảnh face sharp!')

### ID Card: Upload từ Drive + tạo blur

In [ ]:
# Upload ảnh ID card sharp lên Google Drive trước
# rồi copy vào đây:
os.makedirs('datasets/idcard/train/sharp', exist_ok=True)
os.makedirs('datasets/idcard/train/blur', exist_ok=True)
os.makedirs('datasets/idcard/val/sharp', exist_ok=True)
os.makedirs('datasets/idcard/val/blur', exist_ok=True)

# Uncomment dòng dưới nếu đã upload:
# !cp -r '/content/drive/MyDrive/idcard_sharp/'* datasets/idcard/train/sharp/

count = len(glob.glob('datasets/idcard/train/sharp/*'))
print(f'ID Card sharp images: {count}')
if count > 0:
    !python scripts/data_preparation/create_blur_dataset.py \
        --input datasets/idcard/train/sharp \
        --output datasets/idcard/train/blur \
        --task idcard
else:
    print('⚠️ Chưa có ảnh! Upload ảnh ID card vào Drive rồi copy vào')

## 4️⃣ Training
### Chọn task để train (chạy 1 trong 3)

In [ ]:
# ========== TRAIN SCENE DEBLUR ==========
!python basicsr/train.py -opt options/train/train_deblur_scene.yml

In [ ]:
# ========== TRAIN FACE DEBLUR ==========
!python basicsr/train.py -opt options/train/train_deblur_face.yml

In [ ]:
# ========== TRAIN ID CARD DEBLUR ==========
!python basicsr/train.py -opt options/train/train_deblur_idcard.yml

## 5️⃣ Lưu model về Google Drive
**QUAN TRỌNG:** Colab sẽ mất dữ liệu khi disconnect. Luôn save model!

In [ ]:
# Lưu TẤT CẢ experiments về Drive
import shutil, os

save_dir = '/content/drive/MyDrive/DeblurModels'
os.makedirs(save_dir, exist_ok=True)

if os.path.exists('experiments'):
    for exp in os.listdir('experiments'):
        src = f'experiments/{exp}/models'
        if os.path.exists(src):
            dst = f'{save_dir}/{exp}'
            os.makedirs(dst, exist_ok=True)
            for f in os.listdir(src):
                shutil.copy2(f'{src}/{f}', f'{dst}/{f}')
                print(f'  Saved: {dst}/{f}')

print(f'\n[✓] Models saved to: {save_dir}')

## 6️⃣ Test & Inference

In [ ]:
# Test trên ảnh mới
# Upload ảnh blur vào test_images/
os.makedirs('test_images', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Chọn task và model path phù hợp
TASK = 'scene'  # 'face', 'scene', hoặc 'idcard'
MODEL_PATH = 'experiments/DeblurScene_SwinIR/models/net_g_latest.pth'

if os.path.exists(MODEL_PATH):
    !python inference/inference_deblur.py \
        --input test_images \
        --output results \
        --model_path {MODEL_PATH} \
        --task {TASK}
else:
    print(f'⚠️ Chưa có model tại: {MODEL_PATH}')

In [ ]:
# Xem kết quả
import matplotlib.pyplot as plt
import cv2, glob

results = sorted(glob.glob('results/*.png'))[:6]
if results:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, img_path in zip(axes.flat, results):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(os.path.basename(img_path), fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Chưa có kết quả!')